<details><summary style="display:list-item; font-size:16px; color:blue;">Jupyter Help</summary>
    
Having trouble testing your work? Double-check that you have followed the steps below to write, run, save, and test your code!
    
[Click here for a walkthrough GIF of the steps below](https://static-assets.codecademy.com/Courses/ds-python/jupyter-help.gif)

Run all initial cells to import libraries and datasets. Then follow these steps for each question:
    
1. Add your solution to the cell with `## YOUR SOLUTION HERE ## `.
2. Run the cell by selecting the `Run` button or the `Shift`+`Enter` keys.
3. Save your work by selecting the `Save` button, the `command`+`s` keys (Mac), or `control`+`s` keys (Windows).
4. Select the `Test Work` button at the bottom left to test your work.

![Screenshot of the buttons at the top of a Jupyter Notebook. The Run and Save buttons are highlighted](https://static-assets.codecademy.com/Paths/ds-python/jupyter-buttons.png)

**Setup**
Run the following cell to import libraries and helper function.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.datasets as datasets
from torch.utils.data import DataLoader

**CIFAR-10 Testing Set Preprocessing**

In [2]:
data_dir = '/home/ccuser/data'

transform = T.Compose([
    T.Resize(224),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
])

test_dataset  = datasets.CIFAR10(root=data_dir, train=False, download=True, transform=transform)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

Files already downloaded and verified


#### Checkpoint 1/3

Let's load pre-trained weights for a **ResNet18** model (with 11.1M parameters) in this checkpoint.

**A.** Specify the **ResNet18** model architecture from the `torchvision.models` module (aliased as `M`) by using `M.resnet18()`. Since we're loading pre-trained weights, set `weights=None`. Initialize the model to the variable `model_resnet18`. 

**B.** Adapt the final classification layer to output predictions for the 10 classes in the CIFAR-10 dataset using a linear layer. 
- **Note:** **ResNet18** names its last classification layer `fc`, whereas **MobileNet** names it `classification`.

**C.** Save the weights we've pre-trained on the CIFAR-10 training set located in `models/resnet18_cifar10.pt` to the variable `state_dict_resnet18`, and load them into the **ResNet18** model.

**D.** Move the model to the device and set it to evaluation mode.

Don't forget to run the cell and save the notebook before selecting `Test Work`! Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [3]:
import torchvision.models as M

# Move to CPU/GPU device -- DO NOT MODIFY
device = "cuda" if torch.cuda.is_available() else "cpu"

## YOUR SOLUTION HERE ##
# Load ResNet18 and adapt for CIFAR-10 
model_resnet18 = M.resnet18(weights = None)
in_feats = model_resnet18.fc.in_features
model_resnet18.fc = nn.Linear(in_feats, 10)

# Load pre-trained weights fine-tuned on CIFAR-10
state_dict_resnet18 = torch.load("models/resnet18_cifar10.pt",
                                map_location = device,
                                weights_only = True)
model_resnet18.load_state_dict(state_dict_resnet18)

# Move to device and set to evaluation mode
model_resnet18 = model_resnet18.to(device)
model_resnet18 = model_resnet18.eval()

# Show output
from custom_torchinfo import custom_summary
custom_summary(model_resnet18, input_size=(1, 3, 224, 224))

        Layer (type)              Output Shape         Param #
            Conv2d-1         [1, 64, 112, 112]           9,408
       BatchNorm2d-2         [1, 64, 112, 112]             128
              ReLU-3         [1, 64, 112, 112]               0
         MaxPool2d-4           [1, 64, 56, 56]               0
            Conv2d-5           [1, 64, 56, 56]          36,864
       BatchNorm2d-6           [1, 64, 56, 56]             128
              ReLU-7           [1, 64, 56, 56]               0
            Conv2d-8           [1, 64, 56, 56]          36,864
       BatchNorm2d-9           [1, 64, 56, 56]             128
             ReLU-10           [1, 64, 56, 56]               0
       BasicBlock-11           [1, 64, 56, 56]               0
           Conv2d-12           [1, 64, 56, 56]          36,864
      BatchNorm2d-13           [1, 64, 56, 56]             128
             ReLU-14           [1, 64, 56, 56]               0
           Conv2d-15           [1, 64, 56, 56]         

#### Checkpoint 2/3

Create the `predict` function to build our CNN's prediction loop and generate predictions on the testing set images.

**A.** Build the prediction loop as follows:
1. Initialize empty lists for the logits (`all_logits`), predictions (`all_predictions`), and actual labels (`all_labels`).
2. Iterate over the batched data in the dataloader. For each batch:
   - Move them to the device (same as the model).
   - Input through the forward pass to obtain the logits.
   - Convert the logits into probabilities using `softmax()`.
   - Select the predicted class with the highest probability using `argmax()`.
   - Append each output to its respective list (ex., logits to `all_logits`).
3. Join all batched outputs to their respective variables: `y_logits`, `y_pred`, and `y_true`.
4. Return the complete list of outputs in the following order: `y_logits`, `y_pred`, and `y_true`.

**B.** Use the function to generate test predictions from `model_resnet18` from the test dataloader `test_loader`. Save the logits, predicted labels, and true labels to the variables `y_logits_resnet18`, `y_pred_resnet18`, and `y_true`, respectively.

Print the first 10 predicted labels and actual labels.

Don't forget to run the cell and save the notebook before selecting `Test Work`! Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [4]:
## YOUR SOLUTION HERE ##
def predict(model, dataloader, device="cpu"):
    model.eval()
    all_logits = []
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch_X, batch_y in dataloader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            # Forward pass
            logits = model(batch_X)

            # Convert logits to predicted labels
            probs = torch.softmax(logits, dim = 1)
            # Highest probability class
            preds = probs.argmax(dim = 1)
            # Save predictions
            all_logits.append(logits.cpu().numpy())
            all_predictions.append(preds.cpu().numpy())
            all_labels.append(batch_y.cpu().numpy())
    # Join predictions
    y_logits = np.concatenate(all_logits, axis = 0)
    y_pred   = np.concatenate(all_predictions, axis = 0)
    y_true   = np.concatenate(all_labels, axis = 0)

    return y_logits, y_pred, y_true

# Generate test predictions
y_logits_resnet18, y_pred_resnet18, y_true = predict(model_resnet18, test_loader, device = device)

# Show output
print("First 10 predictions:", y_pred_resnet18[:10])
print("First 10 labels:", y_true[:10])

First 10 predictions: [3 8 1 0 6 6 1 6 3 1]
First 10 labels: [3 8 8 0 6 6 1 6 3 1]


#### Checkpoint 3/3

Evaluate the pre-trained ResNet18 on the testing set images by calculating the overall accuracy and creating a classification report.

**A.** Save the overall accuracy to the variable `test_accuracy_resnet18`.

**B.** Save the classification report to the variable `report_resnet18`.
- Add the parameter `target_names=test_dataset.classes` to display the CIFAR-10 class labels in the classification report.

Don't forget to run the cell and save the notebook before selecting `Test Work`! Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [5]:
from sklearn.metrics import accuracy_score, classification_report

## YOUR SOLUTION HERE ##
test_accuracy_resnet18 = accuracy_score(y_true, y_pred_resnet18)
report_resnet18 = classification_report(y_true, y_pred_resnet18, 
                                        target_names = test_dataset.classes)

# Show output
print(f"Accuracy: {test_accuracy_resnet18:.4f}")
print(report_resnet18)

Accuracy: 0.8853
              precision    recall  f1-score   support

    airplane       0.83      0.94      0.88      1000
  automobile       0.98      0.88      0.93      1000
        bird       0.87      0.84      0.85      1000
         cat       0.83      0.72      0.77      1000
        deer       0.90      0.88      0.89      1000
         dog       0.83      0.85      0.84      1000
        frog       0.89      0.94      0.91      1000
       horse       0.90      0.94      0.92      1000
        ship       0.95      0.91      0.93      1000
       truck       0.89      0.95      0.92      1000

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.88     10000
weighted avg       0.89      0.89      0.88     10000



#### Summary

The pre-trained ResNet18 model has an overall accuracy of 88.53%, indicating fairly strong performance. 

**Class-Level Performance**
- Best-performing classes: `automobile`, `ship`, `horse`, and `frog` show strong F1 scores above 0.90.
- Moderate-performing classes: `airplane`, `deer`, `bird`, and `dog` achieve solid F1 scores between 0.84 and 0.89.
- Challenging class: `cat` falls behind with lower recall (0.72) and precision (0.83), resulting in the lowest F1 score of 0.77.

#### Optional Checkpoint

Let's compare the performance of our pre-trained ResNet18 with a pre-trained MobileNet (V3) from the narrative. 

MobileNet (V3) is designed for speed with roughly 1.5M parameters, whereas ResNet18 has about 10x more parameters.

Let's evaluate the pre-trained Mobilenet (V3) using pre-trained weights saved in `"models/mobilenet_cifar10.pt"` and compare the following:
- Classification Performance
- Latency
- Memory (Allocation + Peak During Inference)

In [6]:
import torchvision.models as M

model_mobilenet = M.mobilenet_v3_small(weights = None)
in_feats = model_mobilenet.classifier[-1].in_features
model_mobilenet.classifier[-1] = nn.Linear(in_feats, 10)

state_dict = torch.load("models/mobilenet_cifar10.pt", map_location = device,
                        weights_only = True)

model_mobilenet.load_state_dict(state_dict)
model_mobilenet  = model_mobilenet.to(device).eval()

from custom_torchinfo import custom_summary
custom_summary(model_mobilenet, input_size=(1, 3, 224, 224))

y_logits_mobilenet, y_pred_mobilenet, y_true = predict(
    model=model_mobilenet,
    dataloader=test_loader,
    device=device
)

        Layer (type)              Output Shape         Param #
            Conv2d-1         [1, 16, 112, 112]             432
       BatchNorm2d-2         [1, 16, 112, 112]              32
         Hardswish-3         [1, 16, 112, 112]               0
            Conv2d-4           [1, 16, 56, 56]             144
       BatchNorm2d-5           [1, 16, 56, 56]              32
              ReLU-6           [1, 16, 56, 56]               0
 AdaptiveAvgPool2d-7             [1, 16, 1, 1]               0
            Conv2d-8              [1, 8, 1, 1]             136
              ReLU-9              [1, 8, 1, 1]               0
           Conv2d-10             [1, 16, 1, 1]             144
      Hardsigmoid-11             [1, 16, 1, 1]               0
SqueezeExcitation-12           [1, 16, 56, 56]               0
           Conv2d-13           [1, 16, 56, 56]             256
      BatchNorm2d-14           [1, 16, 56, 56]              32
 InvertedResidual-15           [1, 16, 56, 56]         

In [7]:
test_accuracy_mobilenet = accuracy_score(y_true, y_pred_mobilenet)
report_mobilenet = classification_report(y_true, y_pred_mobilenet, target_names = test_dataset.classes)

In [8]:
import time
import torch

# Grab one image from test loader
batch_X, batch_y = next(iter(test_loader))
x1 = batch_X[0].unsqueeze(0).to(device)

def measure_latency(model, x, iters=30):
    """Return average latency in ms per forward pass."""
    model.eval()

    # Optional warmup
    with torch.inference_mode():
        for _ in range(5):
            _ = model(x)
        if x.device.type == "cuda":
            torch.cuda.synchronize()

    start = time.perf_counter()
    with torch.inference_mode():
        for _ in range(iters):
            _ = model(x)
            if x.device.type == "cuda":
                torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    return (elapsed / iters) * 1e3

def measure_gpu_memory(model, x):
    """Return current and peak GPU memory in MB after one forward pass."""
    if x.device.type != "cuda":
        return None, None

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    model.eval()
    with torch.inference_mode():
        _ = model(x)
    torch.cuda.synchronize()

    current = torch.cuda.memory_allocated() / (1024 ** 2)
    peak = torch.cuda.max_memory_allocated() / (1024 ** 2)
    return current, peak

# Measure latency
latency_mobilenet = measure_latency(model_mobilenet, x1, iters=30)
latency_resnet18 = measure_latency(model_resnet18, x1, iters=30)

# Measure memory
memory_mobilenet = measure_gpu_memory(model_mobilenet, x1)
memory_resnet18 = measure_gpu_memory(model_resnet18, x1)

# Show results
print("Latency Comparison")
print(f"MobileNetV3-Small: {latency_mobilenet:.4f} ms")
print(f"ResNet-18:         {latency_resnet18:.4f} ms")
print("====================================================================")

print("Memory Usage Comparison")
if memory_mobilenet[0] is None:
    print("Running on CPU, so GPU memory metrics are not available.")
else:
    print(f"MobileNetV3-Small: [GPU Memory] Current allocated: {memory_mobilenet[0]:.2f} MB | Peak during forward: {memory_mobilenet[1]:.2f} MB")
    print(f"ResNet-18:         [GPU Memory] Current allocated: {memory_resnet18[0]:.2f} MB | Peak during forward: {memory_resnet18[1]:.2f} MB")
print("====================================================================")

print("Classification Performance Comparison")
print("MobileNetV3-Small Performance:")
print(f"Accuracy: {test_accuracy_mobilenet:.4f}")
print(report_mobilenet)

print("ResNet-18 Performance:")
print(f"Accuracy: {test_accuracy_resnet18:.4f}")
print(report_resnet18)

Latency Comparison
MobileNetV3-Small: 4.2160 ms
ResNet-18:         3.8367 ms
Memory Usage Comparison
MobileNetV3-Small: [GPU Memory] Current allocated: 108.36 MB | Peak during forward: 110.27 MB
ResNet-18:         [GPU Memory] Current allocated: 108.36 MB | Peak during forward: 134.83 MB
Classification Performance Comparison
MobileNetV3-Small Performance:
Accuracy: 0.8492
              precision    recall  f1-score   support

    airplane       0.79      0.92      0.85      1000
  automobile       0.89      0.97      0.93      1000
        bird       0.86      0.75      0.80      1000
         cat       0.81      0.63      0.71      1000
        deer       0.90      0.83      0.86      1000
         dog       0.80      0.78      0.79      1000
        frog       0.78      0.94      0.85      1000
       horse       0.93      0.87      0.90      1000
        ship       0.81      0.95      0.87      1000
       truck       0.97      0.84      0.90      1000

    accuracy                 

In [9]:
# Important: Latency and Time Metrics...

import time

batch_X, batch_y = next(iter(test_loader))
x1 = batch_X[0].unsqueeze(0).to(device)

def measure_latency(model, x, iters=30):
    """Return average latency (ms) per forward pass."""
    model.eval()
    start = time.perf_counter()
    with torch.inference_mode():
        for _ in range(iters):
            _ = model(x)
            if x.device.type == "cuda":
                torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    return (elapsed / iters) * 1e3  # ms

def measure_gpu_memory(model, x):
    """Return current and peak GPU memory in MB after one forward."""
    torch.cuda.empty_cache()                 
    torch.cuda.reset_peak_memory_stats()
    
    model.eval()
    with torch.inference_mode():
        y = model(x) 
        
    torch.cuda.synchronize()
    current = torch.cuda.memory_allocated() / (1024**2)
    peak    = torch.cuda.max_memory_allocated() / (1024**2)
    return current, peak
    
latency_mobilenet = measure_latency(model_mobilenet, x1, iters = 30)
latency_resnet18 = measure_latency(model_resnet18, x1, iters = 30)

latency_mobilenet = measure_latency(model_mobilenet, x1, iters=30)
latency_resnet18  = measure_latency(model_resnet18,  x1, iters=30)

memory_mobilenet = measure_gpu_memory(model_mobilenet, x1)
memory_resnet18 = measure_gpu_memory(model_resnet18, x1)

print("Latency Comparison")
warmup  = measure_latency(model_mobilenet, x1, iters=30)
print("MobileNetV3-Small:", f"{latency_mobilenet:.4f} ms")
print("ResNet-18:",         f"{latency_resnet18:.4f} ms")
print("====================================================================")

print("Memory Usage Comparison")
print(f"MobileNetV3-Small: [GPU Memory] Current allocated: {memory_mobilenet[0]:.2f} MB | Peak during forward: {memory_mobilenet[1]:.2f} MB")
print(f"ResNet-18: [GPU Memory] Current allocated: {memory_resnet18[0]:.2f} MB | Peak during forward: {memory_resnet18[1]:.2f} MB")
print("====================================================================")

print("Classification Performance Comparison")
print("MobileNetV3-Small Performance:")
print(f"Accuracy: {test_accuracy_mobilenet:.4f}")
print(report_mobilenet)
print("ResNet-18 Performance:")
print(f"Accuracy: {test_accuracy_resnet18:.4f}")
print(report_resnet18)

Latency Comparison
MobileNetV3-Small: 4.1943 ms
ResNet-18: 2.5189 ms
Memory Usage Comparison
MobileNetV3-Small: [GPU Memory] Current allocated: 108.36 MB | Peak during forward: 110.27 MB
ResNet-18: [GPU Memory] Current allocated: 108.36 MB | Peak during forward: 134.83 MB
Classification Performance Comparison
MobileNetV3-Small Performance:
Accuracy: 0.8492
              precision    recall  f1-score   support

    airplane       0.79      0.92      0.85      1000
  automobile       0.89      0.97      0.93      1000
        bird       0.86      0.75      0.80      1000
         cat       0.81      0.63      0.71      1000
        deer       0.90      0.83      0.86      1000
         dog       0.80      0.78      0.79      1000
        frog       0.78      0.94      0.85      1000
       horse       0.93      0.87      0.90      1000
        ship       0.81      0.95      0.87      1000
       truck       0.97      0.84      0.90      1000

    accuracy                           0.85  

How does the MobileNet (V3) compare to the ResNet18 model in terms of size, latency, memory allocation, and classification performance?
- Why do you think _"Mobile"_ Net has higher latency? This is most likely due to the model being optimized for CPUs and smartphone devices rather than our GPU device. 

#### Clean Up

In [10]:
import gc, torch

model_mobilenet.to("cpu")
model_resnet18.to("cpu")

del model_mobilenet, model_resnet18
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()